In [1]:
import torch
import sys

from testbed.models import Idefics
sys.path.insert(0, "..")
import config

device = torch.device("cuda:1")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.bfloat16,
).to(device)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [2]:
import pytorch_lightning as pl
import datasets
from torch.utils.data import (
    DistributedSampler,
    BatchSampler,
    SequentialSampler,
    RandomSampler,
)
import os
import sys
from pathlib import Path

from testbed.data import prepare_caption_input, prepare_dataloader, prepare_vqa_input
import config
import exp_settings as setting


class ICVDataModule(pl.LightningDataModule):

    def __init__(self, lmm) -> None:
        super().__init__()
        self.lmm = lmm
        if setting.task == "vqa":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "vqav2"),
                split="train",
                data_dir=config.vqav2_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.task == "caption":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "coco"),
                data_dir=config.karpathy_coco_caption_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        self.query_set = self.dataset.select(
            range(setting.num_query_samples)
        )

    def setup(self, stage: str) -> None:
        if stage == "fit" or stage is None:
            if setting.task == "vqa":
                self.dataset = datasets.load_dataset(
                    os.path.join(config.testbed_dir, "data", "vqav2"),
                    split="train",
                    data_dir=config.vqav2_dir,
                    images_dir=config.coco_dir,
                    trust_remote_code=True,
                )
            elif setting.task == "caption":
                self.dataset = datasets.load_dataset(
                    os.path.join(config.testbed_dir, "data", "coco"),
                    data_dir=config.karpathy_coco_caption_dir,
                    images_dir=config.coco_dir,
                    trust_remote_code=True,
                )
            self.query_set = self.dataset.select(
                range(setting.num_query_samples)
            )

    def collate_fn(self, batch):
        """
        Split batch into full context, in-context examples, query and answer, and process them into model inputs.
        """
        print([b["question_id"] for mb in batch for b in mb])
        if setting.task == "vqa":
            context, images = prepare_vqa_input(
                batch, instruction=setting.vqa_instruction
            )
            # we use the first answer as grounding truth
            answers = [item[-1]["answers"][0]["answer"] for item in batch]
        elif setting.task == "caption":
            context, images = prepare_caption_input(
                batch, instruction=setting.caption_instruction
            )
            answers = [item[-1]["sentences_raw"][0] for item in batch]

        # the last two items (take vqa as an example):
        # [
        #   { "role" : "question"
        #     "content" :  ... },
        #   { "role" : "short answer" }
        # ]
        ice_texts = self.lmm.apply_prompt_template([ctx[:-2] for ctx in context])
        query_texts = self.lmm.apply_prompt_template([ctx[-2:] for ctx in context])

        return {
            "ice_texts": ice_texts,
            "query_texts": query_texts,
            "answers": answers,
            "images": images,
        }

    def train_dataloader(self):
        if self.trainer and self.trainer.world_size > 1:
            samplers = [
                BatchSampler(
                    DistributedSampler(self.dataset, shuffle=False),
                    batch_size=setting.num_shot,
                    drop_last=True,
                ),
                DistributedSampler(self.query_set, shuffle=False),
            ]
        else:
            samplers = [
                BatchSampler(
                    SequentialSampler(self.dataset), batch_size=1, drop_last=True
                ),
                SequentialSampler(self.query_set),
            ]

        return prepare_dataloader(
            [self.dataset, self.query_set],
            setting.batch_size,
            num_per_dataset=[1, 1],
            collate_fn=self.collate_fn,
            samplers=samplers,
            # num_workers=1,
            pin_memory=True,
            shuffle=False,
        )

datamodule = ICVDataModule(lmm)
dataloader = datamodule.train_dataloader()

In [3]:
{
    "ice_prompt": [
        "Instruction: Provide an answer to the question. Use the image to answer.\n",
        "<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>",
        "Question:What is this photo taken looking through? Short answer:net.\n\n",
    ],
    "query_prompt": [
        "<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>",
        "Question:What is this photo taken looking through? Short answer:net.",
    ],
    "query_x": [
        "<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>",
        "Question:What is this photo taken looking through? Short answer:",
    ],
}

{'ice_prompt': ['Instruction: Provide an answer to the question. Use the image to answer.\n',
  '<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>',
  'Question:What is this photo taken looking through? Short answer:net.\n\n'],
 'query_prompt': ['<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>',
  'Question:What is this photo taken looking through? Short answer:net.'],
 'query_x': ['<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>',
  'Question:What is this photo taken looking through? Short answer:']}

In [4]:
inp = next(iter(dataloader))
inputs = {
    "ice_texts": [
        "Instruction: Provide an answer to the question. Use the image to answer.\n<image>Question:What is this photo taken looking through? Short answer:net.\n\n"
    ],
    "query_texts": [
        "<image>Question:What is this photo taken looking through? Short answer:"
    ],
    "answers": ["net"],
    "images": inp["images"],
}
inputs

[458752000, 458752000]


{'ice_texts': ['Instruction: Provide an answer to the question. Use the image to answer.\n<image>Question:What is this photo taken looking through? Short answer:net.\n\n'],
 'query_texts': ['<image>Question:What is this photo taken looking through? Short answer:'],
 'answers': ['net'],
 'images': [[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>,
   <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480>]]}

In [5]:
import torch
from torch import nn
import re


class GlobalICVEncoder(nn.Module):
    def __init__(
        self, lmm_hidden_dim, lmm_layers, alpha_init_value=0.1, **kwargs
    ) -> None:
        super().__init__()

        self.alpha = torch.nn.Parameter(
            torch.full((lmm_layers,), fill_value=alpha_init_value)
        )
        self.icv = torch.nn.Parameter(
            torch.empty(lmm_layers, lmm_hidden_dim).normal_(mean=0.0, std=0.01)
        )

    def hook(self, m, inputs, outputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *rest = outputs
        shift = self.alpha[layer_idx] * self.icv[layer_idx]
        shifted_states = hidden_states + shift[None, None, :].to(device)
        normalized_states = (
            shifted_states
            / shifted_states.norm(dim=-1, keepdim=True)
            * hidden_states.norm(dim=-1, keepdim=True)
        )
        return normalized_states, *rest

In [6]:
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torch.optim as optim
from deepspeed.ops.adam import DeepSpeedCPUAdam
from transformers import get_cosine_schedule_with_warmup

import exp_settings as setting


class ICVModel(pl.LightningModule):
    def __init__(self, lmm, icv_encoder: torch.nn.Module) -> None:
        super().__init__()
        self.lmm = lmm
        self.lmm.requires_grad_(False)
        self.icv_encoder = icv_encoder

    def forward(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        def generate_label_mask(inputs):
            input_ids = inputs["input_ids"]
            batch_size, seq_len = input_ids.shape
            indices = (
                torch.arange(seq_len, device=self.device)
                .unsqueeze(0)
                .expand(batch_size, -1)
            )
            non_pad_mask = input_ids != pad_token_id
            first_non_pad_idx = non_pad_mask.int().argmax(dim=1)

            mask_after_first_non_pad = indices >= first_non_pad_idx.unsqueeze(1)

            separator_pad_mask = mask_after_first_non_pad & ~non_pad_mask
            separator_positions = torch.where(
                separator_pad_mask, indices, torch.full_like(indices, seq_len)
            )
            first_separator_idx = separator_positions.min(dim=1)[0]

            non_answer_mask = (indices >= first_non_pad_idx.unsqueeze(1)) & (
                indices < first_separator_idx.unsqueeze(1)
            )
            return non_pad_mask & ~non_answer_mask

        if hasattr(self.icv_encoder, "hook"):
            hooks = self.lmm.register_forward_hook(
                self.icv_encoder.hook_target,
                self.icv_encoder.hook,
            )
        elif hasattr(self.icv_encoder, "pre_hook"):
            hooks = self.lmm.register_forward_pre_hook(
                self.icv_encoder.hook_target,
                self.icv_encoder.pre_hook,
            )

        # step 1. ICV + query + [PAD] + answer [EOS] forward process
        query_answer = [
            query + pad_token + answer + eos_token
            for query, answer in zip(query_texts, answers)
        ]
        query_images = [img[-setting.num_image_in_query :] for img in images]
        query_inputs = self.lmm.process_input(query_answer, query_images).to(
            self.device
        )
        query_inputs["attention_mask"] = query_inputs["input_ids"] != pad_token_id
        query_outputs = self.lmm.model(
            **query_inputs,
            labels=query_inputs["input_ids"],
        )
        icv_logits = query_outputs["logits"]
        for hook in hooks:
            hook.remove()

        # step 2. ICE + query and answer [EOS] forward process
        full_text = [
            ice + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id
        with torch.no_grad():
            ice_logits = self.lmm.model(**inputs)["logits"]

        # step 3. extract answer logits & calculate kl divergency
        ice_probs = ice_logits[generate_label_mask(inputs)].softmax(dim=-1)
        icv_log_probs = icv_logits[generate_label_mask(query_inputs)].log_softmax(
            dim=-1
        )

        kl_loss = F.kl_div(
            icv_log_probs, ice_probs, reduction="batchmean", log_target=False
        )
        ce_loss = query_outputs["loss"]
        total_loss = kl_loss + setting.ce_loss_weight * ce_loss

        return {
            "kl_loss": kl_loss,
            "ce_loss": ce_loss,
            "loss": total_loss,
        }

    def training_step(self, batch, batch_idx):
        loss_dict = self(**batch)

        self.log_dict(loss_dict, sync_dist=True, prog_bar=True)

        for i, alpha in enumerate(self.icv_encoder.alpha):
            self.log(f"alpha/alpha-{i}", alpha.item())

        return loss_dict["loss"]

    def configure_optimizers(self):
        param_dict = {
            pn: p for pn, p in self.icv_encoder.named_parameters() if p.requires_grad
        }

        alpha_params = [p for n, p in param_dict.items() if "alpha" in n]
        non_alpha_params = [p for n, p in param_dict.items() if not "alpha" in n]

        optim_groups = [
            {"params": non_alpha_params, "lr": setting.icv_lr},
            {"params": alpha_params, "lr": setting.alpha_lr},
        ]

        if "deepspeed" in setting.strategy:
            optimizer = DeepSpeedCPUAdam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )
        else:
            optimizer = optim.Adam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )

        step_batches = self.trainer.estimated_stepping_batches
        warmup_steps = setting.warmup_step
        if isinstance(warmup_steps, float):
            warm_steps = warmup_steps * step_batches
        elif isinstance(warmup_steps, int):
            warm_steps = warmup_steps
        else:
            raise ValueError(
                f"the warm_steps should be int or float, but got {type(warmup_steps)}"
            )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warm_steps, num_training_steps=step_batches
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def on_save_checkpoint(self, checkpoint):
        checkpoint["state_dict"] = {
            k: v
            for k, v in checkpoint["state_dict"].items()
            if not k.startswith("model")
        }
        return checkpoint


[2024-09-14 20:58:25,489] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/jyc/miniconda3/envs/icl/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: und

In [7]:
from testbed.models.model_base import HookType


class AttnAwareEncoder(nn.Module):
    def __init__(
        self, lmm_hidden_dim, lmm_layers, alpha_init_value=0.1, **kwargs
    ) -> None:
        super().__init__()

        self.alpha = torch.nn.Parameter(
            torch.full((lmm_layers,), fill_value=alpha_init_value)
        )
        self.icv = torch.nn.Parameter(
            torch.empty(lmm_layers, lmm_hidden_dim).normal_(mean=0.0, std=0.01)
        )

    @property
    def hook_target(self):
        return HookType.TEXT_MODEL_LAYER

    def pre_hook(self, m, inputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(inputs, tuple):
            inputs = (inputs,)
        hidden_states, *rest = inputs
        shift = self.alpha[layer_idx] * self.icv[layer_idx]
        shifted_states = hidden_states + shift[None, None, :]
        normalized_states = (
            shifted_states
            / shifted_states.norm(dim=-1, keepdim=True)
            * hidden_states.norm(dim=-1, keepdim=True)
        )
        return normalized_states, *rest


torch.manual_seed(426)
icv_encoder = AttnAwareEncoder(4096, 32).to(device, torch.bfloat16)
model = ICVModel(lmm, icv_encoder).to(device, torch.bfloat16)

In [8]:
model(**inputs)


{'kl_loss': tensor(0.8984, device='cuda:1', dtype=torch.bfloat16, grad_fn=<DivBackward0>),
 'ce_loss': tensor(4.3438, device='cuda:1', dtype=torch.bfloat16,
        grad_fn=<NllLossBackward0>),
 'loss': tensor(3.0625, device='cuda:1', dtype=torch.bfloat16, grad_fn=<AddBackward0>)}

In [9]:
# input: ((tensor1,), (tensor2, (tensor3, tensor4)))
icv_encoder.inputs

AttributeError: 'AttnAwareEncoder' object has no attribute 'inputs'

In [34]:
attention_mask

tensor([[[ 0.0205, -0.0571, -0.0554,  ..., -0.0081,  0.0352,  0.0510],
         [-0.0349, -0.0261, -0.0713,  ..., -0.0134,  0.0356,  0.0349],
         [-0.0791, -0.0718, -0.0347,  ..., -0.0043, -0.0037, -0.0400],
         ...,
         [-0.0109, -0.0262,  0.0522,  ...,  0.0011,  0.0420, -0.0835],
         [-0.0312, -0.0093,  0.0168,  ...,  0.0376, -0.0081,  0.0547],
         [ 0.0476, -0.0052,  0.0117,  ..., -0.0168, -0.0337, -0.0325]]],
       device='cuda:1', dtype=torch.bfloat16)